# Thermal noise calculation

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/noise_calculation.ipynb)

Port of SIRIUS technical memo 04: verify the thermal noise added by the simulator against the
theoretical expectation by simulating a **zero-flux** sky, imaging it and comparing the image RMS
with the radiometer equation.

---
## Theory

Following ``casatools.simulator.setnoise(mode="tsys-manual")`` the RMS of one real/imaginary
visibility component on baseline $(1, 2)$ is

$$\sigma = \frac{\sqrt{2}\, k_B\, T_\mathrm{sys}\, 10^{26}}{\eta_c\, \eta_q\, A_\mathrm{eff}\, \sqrt{\Delta\nu\, \Delta t}},\qquad
A_\mathrm{eff} = \eta_a \frac{\pi D_1 D_2}{4},\qquad
T_\mathrm{sys} = T_\mathrm{rx} + T_\mathrm{atm} (1 - \eta_\mathrm{spill}) + T_\mathrm{cmb}$$

and the expected RMS of a naturally weighted Stokes-I image made from $N$ visibilities (2 hands) is
$\sigma_I = 1 / \sqrt{\sum_i w_i}$ with $w_i = 1/\sigma_i^2$.


## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

In [ ]:
from astroviper.processing_functions.simulation.calculate_noise import (
    DEFAULT_NOISE_PARAMS,
    calculate_noise_sigma,
)

DEFAULT_NOISE_PARAMS

## Simulate a zero-flux sky with noise

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout

antenna_xds = read_telescope_layout("vla.d")
n_antenna = antenna_xds.sizes["antenna_name"]
n_time, time_delta = 10, 3600.0
n_channel, channel_width = 3, 0.01e9
phase_center = SkyCoord(ra="19h59m28.5s", dec="+40d44m01.5s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

result = simulate_processing_set(
    ps_store="noise_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params={
        "time_start": "2019-10-03T19:00:00.000",
        "time_delta": time_delta,
        "n_samples": n_time,
    },
    frequency_params={
        "freq_start": 3e9,
        "freq_delta": 0.4e9,
        "n_channels": n_channel,
        "channel_width": channel_width,
    },
    polarization=["RR", "LL"],
    point_source_flux=np.zeros((1, 1, 1, 4)),
    point_source_ra_dec=phase_center_ra_dec[None, :, :],
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=[airy_disk_model("vla")],
    beam_model_map=np.zeros(n_antenna, dtype=int),
    noise_params={"random_seed": 1},  # defaults: t_receiver = 50 K, ...
    n_time_chunks=2,
    n_frequency_chunks=3,
    overwrite=True,
)

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set (coordinates,
dimensions, dtypes and attributes of the main, antenna and field/source datasets) against the
MSv4 schema.  ``simulate_processing_set`` runs this check itself (``check_schema=True``) and logs
a warning on problems; here it is run explicitly so that the result is visible.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set("noise_sim.ps.zarr")
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"
# the same check works on a single MSv4 (the checker dispatches on the ``type`` attribute)
print(check_datatree(ps_xdt[result["ms_name"]]))

## Compare the visibility noise with the theoretical sigma

In [ ]:
from xradio.measurement_set import load_processing_set

from astroviper.utils.measurement_set_tools import baseline_antenna_pairs

ms_xds = load_processing_set("noise_sim.ps.zarr")[result["ms_name"]].ds
antenna1, antenna2 = baseline_antenna_pairs(n_antenna)
sigma_theory = calculate_noise_sigma(
    np.full(n_antenna, 24.5),
    antenna1,
    antenna2,
    channel_width,
    time_delta,
    DEFAULT_NOISE_PARAMS,
)
print(
    "theoretical sigma per baseline (all equal for a homogeneous array):",
    sigma_theory[0],
    "Jy",
)
print("measured std of Re(V):", ms_xds.VISIBILITY.values.real.std(), "Jy")
print("measured std of Im(V):", ms_xds.VISIBILITY.values.imag.std(), "Jy")
print(
    "WEIGHT = 1/sigma^2:",
    ms_xds.WEIGHT.values[0, 0, 0, 0],
    "vs",
    1 / sigma_theory[0] ** 2,
)

## Image and compare the image RMS with theory

In [ ]:
from xradio.image import load_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store,
    image_store,
    image_size,
    cell_size_arcsec,
    niter=0,
    polarization_coords=("I",),
    n_chunks=2,
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_params = {
        "image_size": list(image_size),
        "cell_size": np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": list(polarization_coords),
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    }
    iteration_control = {
        "niter": niter,
        "nmajor": -1 if niter > 0 else 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.1,
    }
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params=iteration_control,
        gridder="prolate_spheroidal",
        deconvolver="hogbom_many_threads",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=n_chunks,
        overwrite=True,
        restore=niter > 0,
    )
    return load_image(image_store)


def show_image(
    img_xds, variable="SKY_RESIDUAL", frequency=0, polarization=0, title=None, vmax=None
):
    """Plot one plane of an AstroVIPER image with l/m in arcsec."""
    plane = (
        img_xds[variable]
        .isel(time=0, frequency=frequency, polarization=polarization)
        .values
    )
    extent = (
        np.array(
            [
                img_xds.l.values[0],
                img_xds.l.values[-1],
                img_xds.m.values[0],
                img_xds.m.values[-1],
            ]
        )
        / ARCSEC_TO_RAD
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(plane.T, origin="lower", extent=extent, cmap="viridis", vmax=vmax)
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
    ax.set_title(
        title
        or f"{variable} channel {frequency} ({img_xds.frequency.values[frequency] / 1e9:.3f} GHz)"
    )
    fig.colorbar(im, ax=ax, label="Jy/beam")
    return fig

In [ ]:
img_xds = image_simulation(
    "noise_sim.ps.zarr",
    "noise_sim.img.zarr",
    image_size=[400, 400],
    cell_size_arcsec=5.0,
    niter=0,
    n_chunks=3,
)
show_image(img_xds, frequency=0, title="noise-only dirty image (channel 0)")
plt.show()
sky = img_xds.SKY_RESIDUAL.isel(time=0, polarization=0).values
weight_sum = ms_xds.WEIGHT.values.sum(axis=(0, 1, 3))  # per channel, both hands
expected = 1 / np.sqrt(weight_sum)
measured = sky.std(axis=(1, 2))
for ch in range(n_channel):
    print(
        f"channel {ch}: measured image rms {measured[ch]:.4e} Jy/beam, theory {expected[ch]:.4e} Jy/beam, ratio {measured[ch] / expected[ch]:.3f}"
    )

## Clean up

In [ ]:
import shutil

shutil.rmtree("noise_sim.ps.zarr", ignore_errors=True)
shutil.rmtree("noise_sim.img.zarr", ignore_errors=True)
viper_client.close()